In [68]:
# Bayesian Optimization with Gaussian Process Surrogate Model

# This notebook implements a closed-loop Bayesian optimization framework using Gaussian Process Regression (GPR) for multi-objective optimization of transparency and sheet resistance.

# The workflow includes:
# - surrogate model training
# - leave-one-out cross-validation (LOOCV)
# - Pareto-based acquisition (LCB)
# - iterative experiment proposal
# - uncertainty-aware stopping criteria

In [69]:
# =========================================
# Bayesian Optimization with Gaussian Process
# Pareto Front Multi-objective Optimization
# Install & Imports
# =========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut

import os

In [51]:
# =========================
# CONFIGURATION
# =========================

CSV_FILE = "data_GPR.csv"
PARAM_BOUNDS = {
    "power": (0.05, 1.0),
    "flow_rate": (100, 300),
    "PVP_conc": (0.0, 0.5)
}

TARGET = {
    "transmittance": 74.0,
    "resistance": 15.0
}

N_CANDIDATES = 20000
N_SELECT = 3
N_LOCAL = 2
LOCAL_PCT = 0.05
KAPPA = 1.0

# normalized scaling for objective space distance
DIST_SCALE = np.array([10.0, 5.0])

In [52]:
# =========================
# LOAD EXPERIMENTAL DATA
# =========================

df = pd.read_csv(CSV_FILE, sep=';')

df = df[(df["resistance"] > 0) & (df["resistance"] < 5000)]

X = df[["power", "flow_rate", "PVP_conc"]].values
y_trans = df["transmittance"].values
y_res = df["resistance"].values

print("Dataset size:", len(df))


Dataset size: 127


In [ ]:
# =========================
# VALIDATION (LOOCV)
# =========================

from sklearn.model_selection import LeaveOneOut
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RationalQuadratic, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
import numpy as np

def loocv_rmse(X, y):
    """
    Leave-One-Out Cross-Validation RMSE for Gaussian Process surrogate model.
    Returns RMSE in original physical units.
    """

    loo = LeaveOneOut()
    squared_errors = []

    for train_idx, test_idx in loo.split(X):

        # split
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # -------------------------
        # SCALE INPUT FEATURES
        # -------------------------
        x_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train)
        X_test_scaled = x_scaler.transform(X_test)

        # -------------------------
        # SCALE OUTPUT
        # -------------------------
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

        # -------------------------
        # GAUSSIAN PROCESS MODEL
        # -------------------------
        kernel = C(1.0, (1e-3, 1e5)) * RationalQuadratic(
            length_scale=1.0,
            alpha=1.0,
            length_scale_bounds=(1e-3, 1e3)
        )

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-6,
            normalize_y=False,
            n_restarts_optimizer=3
        )

        gp.fit(X_train_scaled, y_train_scaled)

        # prediction (scaled space)
        pred_scaled = gp.predict(X_test_scaled)

        # back to physical units
        pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1))[0, 0]

        # squared error in real units
        squared_errors.append((pred - y_test[0]) ** 2)

    rmse = np.sqrt(np.mean(squared_errors))
    return rmse

rmse_trans = loocv_rmse(X, y_trans)
rmse_res = loocv_rmse(X, y_res)

print("=================================")
print("LOOCV VALIDATION")
print("=================================")
print(f"Transmittance RMSE: {rmse_trans:.4f}")
print(f"Resistance RMSE: {rmse_res:.4f}")
print("=================================")

The LOOCV results show that the surrogate model achieves good predictive accuracy for transmittance, while the prediction error for sheet resistance is higher. This discrepancy is attributed to the inherently higher variability and nonlinear dependence of electrical resistance on processing parameters. Importantly, the Gaussian Process model provides uncertainty estimates, which are explicitly incorporated into the acquisition function, ensuring robust optimization despite increased prediction uncertainty.

In [56]:
# =========================
# SAMPLING
# =========================

def sample_candidates(bounds, n):
    keys = list(bounds.keys())
    X = np.zeros((n, len(keys)))

    for i, k in enumerate(keys):
        X[:, i] = np.random.uniform(bounds[k][0], bounds[k][1], n)

    return X, keys


# =========================
# PARETO FILTER
# =========================

def pareto_mask(costs):
    N = len(costs)
    mask = np.ones(N, dtype=bool)

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            if (costs[j,0] >= costs[i,0] and costs[j,1] <= costs[i,1]) and \
               (costs[j,0] > costs[i,0] or costs[j,1] < costs[i,1]):
                mask[i] = False
                break
    return mask


# =========================
# LOCAL EXPLOITATION
# =========================

def local_variation(x):
    lo = np.array([PARAM_BOUNDS[k][0] for k in PARAM_BOUNDS])
    hi = np.array([PARAM_BOUNDS[k][1] for k in PARAM_BOUNDS])
    span = hi - lo

    noise = (np.random.uniform(-1,1,len(x)) * LOCAL_PCT) * span
    return np.clip(x + noise, lo, hi)

In [66]:
# =========================
# BAYESIAN OPTIMIZATION STEP
# =========================

def run_iteration(X, y_t, y_r):

    # -------------------------
    # SCALE INPUTS
    # -------------------------
    x_scaler = StandardScaler()
    Xs = x_scaler.fit_transform(X)

    yt_scaler = StandardScaler()
    yr_scaler = StandardScaler()

    yt = yt_scaler.fit_transform(y_t.reshape(-1,1)).ravel()
    yr = yr_scaler.fit_transform(y_r.reshape(-1,1)).ravel()

    # -------------------------
    # GAUSSIAN PROCESS MODELS
    # -------------------------
    kernel = C(1.0, (1e-3, 1e5)) * RationalQuadratic()

    gp_t = GaussianProcessRegressor(kernel=kernel, alpha=1e-6)
    gp_r = GaussianProcessRegressor(kernel=kernel, alpha=1e-6)

    gp_t.fit(Xs, yt)
    gp_r.fit(Xs, yr)

    # -------------------------
    # SAMPLE CANDIDATES
    # -------------------------
    Xcand, keys = sample_candidates(PARAM_BOUNDS, N_CANDIDATES)
    Xc = x_scaler.transform(Xcand)

    mu_t_s, sigma_t_s = gp_t.predict(Xc, return_std=True)
    mu_r_s, sigma_r_s = gp_r.predict(Xc, return_std=True)

    # -------------------------
    # BACK TO PHYSICAL UNITS
    # -------------------------
    mu_t = yt_scaler.inverse_transform(mu_t_s.reshape(-1,1)).ravel()
    mu_r = yr_scaler.inverse_transform(mu_r_s.reshape(-1,1)).ravel()

    sigma_t = sigma_t_s * yt_scaler.scale_[0]
    sigma_r = sigma_r_s * yr_scaler.scale_[0]

    # -------------------------
    # ACQUISITION (LCB)
    # -------------------------
    eff_t = mu_t - KAPPA * sigma_t
    eff_r = mu_r + KAPPA * sigma_r

    acq = np.vstack([eff_t, eff_r]).T
    p_mask = pareto_mask(acq)

    Xp = Xcand[p_mask]

    Yp = np.vstack([mu_t[p_mask], mu_r[p_mask]]).T

    # =========================
    # DISTANCE TO IDEAL POINT
    # =========================

    if len(Yp) > 0:

        tgt = np.array([100.0, 0.0])

        scale = np.array([
            10.0,
            np.std(y_r)
        ])

        dists = np.linalg.norm((Yp - tgt) / scale, axis=1)
        min_distance = np.min(dists)

    else:
        min_distance = float('inf')

    # -------------------------
    # NORMALIZED UNCERTAINTY (consistent with distance)
    # -------------------------
    sigma_norm = np.sqrt(
        (sigma_t / 10.0)**2 +
        (sigma_r / np.std(y_r))**2
    )

    if np.any(p_mask):
        mean_sigma = np.mean(sigma_norm[p_mask])
    else:
        mean_sigma = np.mean(sigma_norm)

    # -------------------------
    # SELECTION
    # -------------------------
    selected = Xp[:N_SELECT] if len(Xp) > 0 else np.empty((0, X.shape[1]))

    new_points = []
    for x in selected:
        new_points.append(x)
        new_points.append(local_variation(x))

    new_points = np.vstack(new_points) if len(new_points) > 0 else np.empty((0,X.shape[1]))

    return new_points, min_distance, mean_sigma

In [ ]:
# =========================
# RUN ONE ITERATION
# =========================

new_points, min_dist, mean_sigma = run_iteration(X, y_trans, y_res)


print("New points:", len(new_points))
print("Min normalized distance:", min_dist)
print("Mean uncertainty:", mean_sigma)


In [60]:
pd.DataFrame(new_points, columns=PARAM_BOUNDS.keys()).to_csv(
    "proposed_points.csv",
    index=False
)